# SIDD Medium 数据划分（默认不覆盖 CSV）

按顺序运行单元格。新会话无需先运行其他 Notebook。

### 1. 环境及路径

In [ ]:
import csv
from collections import Counter
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "src").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("请从项目根目录或 notebooks/ 目录启动 Jupyter")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("项目目录：", PROJECT_ROOT)
sidd_root = PROJECT_ROOT / "data" / "SIDD_Medium_Srgb" / "Data"
csv_path = PROJECT_ROOT / "data" / "splits" / "sidd_split.csv"
if not sidd_root.is_dir():
    raise FileNotFoundError(sidd_root)
print("SIDD 数据：", sidd_root)
print("现有 CSV：", csv_path)


### 2. 固定场景划分

In [ ]:
TRAIN_SCENES = {"001", "002", "003", "004", "005", "006"}
VAL_SCENES = {"007", "010"}
TEST_SCENES = {"008", "009"}
assert not (TRAIN_SCENES & VAL_SCENES or TRAIN_SCENES & TEST_SCENES or VAL_SCENES & TEST_SCENES)

def split_for(scene_id):
    for split, scene_ids in (("train", TRAIN_SCENES), ("val", VAL_SCENES), ("test", TEST_SCENES)):
        if scene_id in scene_ids:
            return split
    raise ValueError(f"未知 scene_id：{scene_id}")


### 3. 扫描 NOISY/GT 并生成内存记录

In [ ]:
rows = []
scene_dirs = sorted(path for path in sidd_root.iterdir() if path.is_dir())
for scene_dir in scene_dirs:
    parts = scene_dir.name.split("_")
    if len(parts) != 7:
        raise ValueError(f"目录格式异常：{scene_dir.name}")
    scene_instance, scene_id, smartphone, iso, shutter, color_temp, brightness = parts
    for noisy in sorted(scene_dir.glob("*_NOISY_SRGB_*.PNG")):
        gt = noisy.with_name(noisy.name.replace("_NOISY_SRGB_", "_GT_SRGB_"))
        if not gt.is_file():
            raise FileNotFoundError(gt)
        rows.append({
            "split": split_for(scene_id), "scene_id": scene_id,
            "scene_instance": scene_instance, "smartphone": smartphone,
            "iso": iso, "shutter": shutter, "color_temp": color_temp,
            "brightness": brightness,
            "noisy_path": str(noisy.relative_to(PROJECT_ROOT)),
            "gt_path": str(gt.relative_to(PROJECT_ROOT)),
        })
print("场景实例：", len(scene_dirs))
print("图像配对数：", len(rows))
print("划分统计：", dict(Counter(row["split"] for row in rows)))


### 4. 场景隔离及现有 CSV 一致性检查

In [ ]:
splits = {
    k: {r["scene_id"] for r in rows if r["split"] == k}
    for k in ("train", "val", "test")
}
assert splits["train"].isdisjoint(splits["val"])
assert splits["train"].isdisjoint(splits["test"])
assert splits["val"].isdisjoint(splits["test"])
assert len(rows) == len({(r["noisy_path"], r["gt_path"]) for r in rows})
print("场景隔离检查通过")

FIELDS = ["split", "scene_id", "scene_instance", "smartphone", "iso", "shutter", "color_temp", "brightness", "noisy_path", "gt_path"]
if csv_path.is_file():
    with csv_path.open(encoding="utf-8-sig", newline="") as stream:
        old_rows = list(csv.DictReader(stream))
    def normalized(r):
        return tuple(r[key].replace("\\", "/") if key.endswith("_path") else r[key] for key in FIELDS)
    matches = [normalized(r) for r in rows] == [normalized(r) for r in old_rows]
    print("现有 CSV 数量：", len(old_rows))
    print("现有 CSV 与扫描结果（含索引顺序）一致：", matches)
    if not matches:
        print("警告：CSV 与扫描结果不一致！不要覆盖已有 CSV 或缓存。")
else:
    print("CSV 尚不存在：", csv_path)


### 5. 可选写入（默认禁用）

In [ ]:
WRITE_CSV = False
ALLOW_OVERWRITE = False

if not WRITE_CSV:
    print("只读检查模式：未写入 CSV")
else:
    if csv_path.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError("现有 CSV 受到保护；如确需覆盖，请同时设置 ALLOW_OVERWRITE=True")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    print("已生成：", csv_path)
    print("重要：若 CSV 顺序变化，原训练/验证缓存必须重新匹配或重建！")
